# rail-extract: GPU runs on Kaggle

Settings first (right-hand panel): **Accelerator: GPU T4 x2**, **Internet: on**.

Then **Save Version > Save & Run All**. It keeps running if you close the browser.
When it ends, download `rail_extract_output.zip` from the Output tab.

See `docs/KAGGLE.md`.


In [ ]:
# What to run: "speed" (bf16 vs fp16, about 20 minutes) or "overfit" (about 75 minutes).
STAGE = "speed"
REF = "main"  # branch or commit of the public repository
REPO = "https://github.com/cardohso/rail-extract-lora.git"

In [ ]:
import os
import shlex
import signal
import subprocess
import sys


def run(cmd, env=None):
    """Run a shell command, stream its output, stop the notebook if it fails. If the cell is
    interrupted, the command is stopped too, so it never keeps holding the GPU."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, env={**os.environ, **(env or {})}, start_new_session=True
    )
    try:
        code = proc.wait()
    except BaseException:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait()
        raise
    if code:
        sys.exit(f"command failed with exit code {code}: {cmd}")


def gpu_used_mib():
    out = subprocess.run(
        "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
        shell=True,
        capture_output=True,
        text=True,
    ).stdout.split()
    return int(out[0])


run("nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv")
if gpu_used_mib() > 500:
    sys.exit(
        "The GPU is already in use by another process (an earlier run in this session). "
        "Use Run > Restart Session, or better: Save Version > Save & Run All."
    )
run("pip install -q uv")
os.chdir("/kaggle/working")  # a folder left by an earlier run is removed first
run("rm -rf /kaggle/working/repo")
run(f"git clone -q {shlex.quote(REPO)} /kaggle/working/repo")
os.chdir("/kaggle/working/repo")
run(f"git checkout -q {shlex.quote(REF)} && git log --oneline -1")
run("uv sync --frozen --group train")
PY = "uv run --frozen --group train python"
GPU = {"CUDA_VISIBLE_DEVICES": "0", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
run(
    f"{PY} -c \"import torch; assert torch.cuda.is_available(), 'no GPU visible'; "
    f'print(torch.__version__, torch.cuda.get_device_name(0))"',
    GPU,
)

In [ ]:
RUNS = "/kaggle/working/runs"
if STAGE == "overfit":
    run(f"{PY} -m rail_extract.train_lora --overfit --out {RUNS}/overfit --ckpt-every 40", GPU)
elif STAGE == "speed":
    # same 10 steps of C1 in each precision; compare seconds_per_step and loss
    for prec in ("bf16", "fp16"):
        run(
            f"{PY} -m rail_extract.train_lora --config C1 --precision {prec} --stop-after 10 "
            f"--out {RUNS}/speed-{prec}",
            GPU,
        )
    for prec in ("bf16", "fp16"):
        run(
            f"{PY} -c \"import json; d=json.load(open('{RUNS}/speed-{prec}/train_log.json')); "
            f"print('{prec}', d['seconds_per_step'], 's/step, peak GB', d['peak_gpu_gb'], "
            f"'losses', d['losses'])\""
        )
else:
    sys.exit(f"unknown stage {STAGE}")

In [ ]:
# Everything except the checkpoints goes in one zip.
run(
    "cd /kaggle/working && zip -qr rail_extract_output.zip runs -x '*/training_state.pt' && "
    "ls -la rail_extract_output.zip"
)
run("rm -rf /kaggle/working/repo")  # keep only the outputs in the Output tab